# COMP3122 – Lab 07 – Classification by Similarity
### Week 9 · Chapter 7: KNN

**Applied Data Science**

**This lab is graded.** It covers Chapter 7 in the order the chapter covers it.

**How it works:** each task has a cell for your answer, followed by a cell that checks it. Run the check cell and it prints `PASS`, or shows you what it expected.

**Derive your answers; do not type them.** A label or an accuracy typed from the chapter passes the same check as one you computed. Several tasks check how you got there as well – Task 4.1 checks which flowers landed in your test set, and Tasks 6.2 and 6.4 read what your scaler learned – but mostly this is on you.

**Three tasks ask you to write down a prediction before you run anything** – Tasks 2.4, 5.1 and 6.3. Do that. At least one of them surprises most people.

**Part 6 is the one to slow down for.** Task 6.3 runs the chapter's scaling argument on 300 people you have not seen before.

**Nothing in this lab needs the internet.**

Work in order. The setup cell builds `X_six`, `y_six`, `new_flower`, `X`, `y`, `customers`, `new_customer` and `members`, and defines the checkers, so run it first. **Do not reassign those names** – build your answers into the variables each task asks for. If a check will not pass after five minutes, ask.

---
## Setup – run this first

In [ ]:
# Run this cell once, before you start. It builds the data and defines the checkers.
import os
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")   # explain: silences a harmless CPU-count warning on the course server
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# ---- the six flowers from Chapter 7 §7.2 (petal length, petal width, in cm) ----
X_six = np.array([[1.4, 0.2], [1.3, 0.4], [1.7, 0.5],
                  [4.1, 1.0], [4.6, 1.5], [3.9, 1.2]])
y_six = np.array(["setosa", "setosa", "setosa", "versicolor", "versicolor", "versicolor"])
new_flower = np.array([4.4, 1.4])

# ---- the Iris dataset: 150 flowers, 4 features, 3 species. No download. -----------
iris = load_iris(as_frame=True)
X = iris.data     # a DataFrame: 150 rows, 4 feature columns
y = iris.target   # a Series: species codes 0, 1, 2

# ---- the four customers from §7.6, and the new one ------------------------------
customers = pd.DataFrame({"income_usd": [30000.0, 36000.0, 82000.0, 90000.0],
                          "age":        [60.0, 58.0, 25.0, 30.0],
                          "premium":    ["yes", "yes", "no", "no"]})
new_customer = pd.DataFrame({"income_usd": [34000.0], "age": [27.0]})

# ---- 300 club members, for the scaling flip on data you have not seen -----------
# Generated from a fixed seed, so every student gets exactly the same 300 rows.
# Whether a person joined depends on AGE. Income is unrelated to joining.
_rng = np.random.default_rng(3122)
_income = _rng.normal(60000, 15000, 300).round(-2)
_age = _rng.integers(18, 76, 300)
_joined = (_age + _rng.normal(0, 6, 300) > 45)
members = pd.DataFrame({"income_usd": _income, "age": _age,
                        "joined": np.where(_joined, "yes", "no")})

# Untouched copies, so the Task 7.3 guard can tell whether anything was reassigned.
X_SIX_AT_START, Y_SIX_AT_START = X_six.copy(), y_six.copy()
X_AT_START, Y_AT_START = X.copy(), y.copy()
CUSTOMERS_AT_START, MEMBERS_AT_START = customers.copy(), members.copy()

# The rows a correct Task 4.1 split puts in the test set, from the untouched data.
TEST_ROWS_41 = sorted(train_test_split(X_AT_START, test_size=0.2, random_state=20)[1].index)


def check(label, got, want):
    """Compare your answer with the expected value. Prints PASS or NOT YET.

    `got` may be a value, or a zero-argument function producing one. The second
    form is used wherever the check has to reach into your answer first – read an
    attribute, round a number, index into an array. Before the answer is filled in
    that would raise, and a traceback where a hint should be is not helpful, so it
    is caught and reported as NOT YET instead.
    """
    if callable(got):
        try:
            got = got()
        except Exception as e:
            print(f"NOT YET  {label}")
            print(f"         your answer raised {type(e).__name__}: {e}")
            return

    def _fail(why=None):
        print(f"NOT YET  {label}")
        if why:
            print(f"         {why}")
        print(f"         you gave: {got!r}")
        print(f"         expected: {want!r}")

    PANDAS = (pd.Series, pd.DataFrame)
    if isinstance(want, PANDAS) or isinstance(got, PANDAS):
        if type(got) is not type(want):
            return _fail(f"wrong type – got {type(got).__name__}, expected {type(want).__name__}")
        if list(got.index) != list(want.index):
            return _fail("the row labels do not match")
        if isinstance(want, pd.DataFrame) and list(got.columns) != list(want.columns):
            return _fail("the column labels do not match")
        try:
            ok = bool(got.equals(want))
        except Exception:
            ok = False
    else:
        g, w = np.asarray(got), np.asarray(want)
        try:
            if g.shape == w.shape and g.dtype.kind in "fc" and w.dtype.kind in "fc":
                ok = bool(np.allclose(g, w))   # decimals: allow a tiny rounding difference
            else:
                ok = bool(np.array_equal(g, w))
        except Exception:
            ok = False

    if ok:
        print(f"PASS  {label}")
    else:
        _fail()


def error_message(fn):
    """Run fn() and return the whole error message it raised, or 'no error'.

    Task 1.2 uses it to show the full message without stopping the notebook.
    """
    try:
        fn()
    except Exception as e:
        return f"{type(e).__name__}: {e}"
    return "no error"


def whole(v):
    """A count or a position must be a whole number. 4 and 4.0 count as 4; 1.9 stays 1.9
    and does not pass. (Turning 1.9 into 1 would let a wrong answer through.)"""
    if isinstance(v, (bool, np.bool_)):
        return v
    if isinstance(v, (int, np.integer)):
        return int(v)
    if isinstance(v, (float, np.floating)) and float(v).is_integer():
        return int(v)
    return v


def raises(fn):
    """Run fn() and return the name of the exception it raised, or 'no error'.

    Several tasks ask you to find out what a wrong line actually does. Guessing
    is not the same as knowing, and this is how you check.
    """
    try:
        fn()
    except Exception as e:
        return type(e).__name__
    return "no error"


print("Ready. X_six:", X_six.shape, " Iris X:", X.shape, " customers:", customers.shape,
      " members:", members.shape)
import sklearn
print("scikit-learn", sklearn.__version__, " pandas", pd.__version__, " numpy", np.__version__)


---
## Part 1 – Regression or classification?  ·  §7.1

Regression predicts a **number**. Classification predicts a **category** from a fixed list.
The first question about any new problem is which of the two it is.

**Task 1.1.** For each question, decide whether it is a regression problem or a
classification problem. Answer with the strings `'regression'` or `'classification'`.

- `t11_a` – how many minutes a bus will be late
- `t11_b` – whether an email is spam or not spam
- `t11_c` – which of three species a flower is
- `t11_d` – the price a used car will sell for

In [1]:
t11_a = 'regression'
t11_b = 'classification'
t11_c = 'classification'
t11_d = 'regression'

In [2]:
check('1.1 a bus minutes late',  t11_a, 'regression')
check('1.1 b spam or not',       t11_b, 'classification')
check('1.1 c which species',     t11_c, 'classification')
check('1.1 d used-car price',    t11_d, 'regression')

NameError: name 'check' is not defined

**Task 1.2.** The Iris species codes are names, not quantities.

`iris.target_names` turns a code into a species name. Store the **names** of the codes
`[2, 0, 1]`, in that order, as a list of strings in `t12_names`.

Hint: `iris.target_names[[2, 0, 1]]` gives an array; `list(...)` turns it into a list.

In [ ]:
t12_names = list(iris.target_names[[2, 0, 1]])

In [ ]:
check('1.2 names of codes 2, 0, 1', lambda: [str(n) for n in t12_names], ['virginica', 'setosa', 'versicolor'])

---
## Part 2 – The nearest-neighbor idea, by hand  ·  §7.2

Six flowers (`X_six`, `y_six`) and one new flower (`new_flower`). KNN in three steps:
**measure** every distance, **sort** and take the K nearest, **vote**.

**Task 2.1.** Compute the Euclidean distance from `new_flower` to each of the six flowers,
**in one line of NumPy** (broadcasting, then a sum along `axis=1`, then a square root).
Store the six distances, **rounded to 2 decimals**, in `t21_dist`.

In [ ]:
t21_dist = np.sqrt(((X_six - new_flower) ** 2).sum(axis=1)).round(2)

In [ ]:
check('2.1 six distances', lambda: np.asarray(t21_dist, dtype=float).round(2).tolist(),
      [3.23, 3.26, 2.85, 0.5, 0.22, 0.54])

**Task 2.2.** Find the **positions** of the three nearest flowers, nearest first, with
`np.argsort`. Store them as a list of integers in `t22_nearest3`, and their labels
(from `y_six`) as a list of strings in `t22_labels`.

In [ ]:
t22_nearest3 = [int(i) for i in np.argsort(t21_dist)[:3]]
t22_labels = [str(y_six[i]) for i in t22_nearest3]

In [ ]:
check('2.2 nearest three positions', lambda: [whole(i) for i in t22_nearest3], [4, 3, 5])
check('2.2 their labels', lambda: [str(s) for s in t22_labels], ['versicolor', 'versicolor', 'versicolor'])

**Task 2.3.** Now let scikit-learn do Steps 1-3. Fit a `KNeighborsClassifier` with K = 3
on `X_six` and `y_six`, and predict the species of `new_flower`.

Remember that `predict()` wants a **table**: one flower is a table with one row.
Store the predicted species (a string) in `t23_pred`.

In [ ]:
m23 = KNeighborsClassifier(n_neighbors=3).fit(X_six, y_six)
t23_pred = m23.predict([new_flower])[0]

In [ ]:
check('2.3 K = 3 prediction', lambda: str(t23_pred), 'versicolor')

**Task 2.4 – predict first.** Before you run anything, write down what you expect K = 6
to predict for `new_flower`. With six flowers, K = 6 means **every** flower votes.

Then fit the model with K = 6 and store the prediction in `t24_pred`. Store `True` in
`t24_is_tie` if the vote was tied, or `False` if it was not.

In [ ]:
# Prediction before running: 3 setosa and 3 versicolor tie; alphabetical tie-break predicts 'setosa'.
m24 = KNeighborsClassifier(n_neighbors=6).fit(X_six, y_six)
t24_pred = m24.predict([new_flower])[0]
t24_is_tie = True

In [ ]:
check('2.4 K = 6 prediction', lambda: str(t24_pred), 'setosa')
check('2.4 was the vote tied?', t24_is_tie, True)

---
## Part 3 – Look at Iris before you model  ·  §7.3

`X` (150 × 4) and `y` (150 species codes) are built in the setup cell.

**Task 3.1.** Record what a first look shows:

- `t31_shape` – the shape of `X`
- `t31_counts` – how many flowers of each species, as a list in code order 0, 1, 2
  (hint: `y.value_counts().sort_index()`)

In [ ]:
t31_shape = X.shape
t31_counts = y.value_counts().sort_index().tolist()

In [ ]:
check('3.1 shape of X', lambda: tuple(t31_shape), (150, 4))
check('3.1 flowers per species', lambda: list(t31_counts), [50, 50, 50])

**Task 3.2.** The pairplot shows setosa as a separate group in the petal panels. Prove it
with two numbers:

- `t32_setosa_max` – the **largest** petal length among setosa flowers (code 0)
- `t32_others_min` – the **smallest** petal length among the other two species

Use the column `X['petal length (cm)']` and a boolean mask on `y`.

In [ ]:
t32_setosa_max = X['petal length (cm)'][y == 0].max()
t32_others_min = X['petal length (cm)'][y != 0].min()

In [ ]:
check('3.2 largest setosa petal length', lambda: float(t32_setosa_max), 1.9)
check('3.2 smallest petal length of the others', lambda: float(t32_others_min), 3.0)
check('3.2 so there is a clear gap', lambda: t32_setosa_max < t32_others_min, True)

---
## Part 4 – KNN with scikit-learn  ·  §7.4

Split, fit, predict, score – the Chapter 6 contract with a new model.

**Task 4.1.** Split `X` and `y` into training and test sets with `test_size=0.2` and
`random_state=20`. Store the four pieces in `X41_train`, `X41_test`, `y41_train`,
`y41_test`, in the usual order.

In [ ]:
X41_train, X41_test, y41_train, y41_test = train_test_split(X, y, test_size=0.2, random_state=20)

In [ ]:
check('4.1 training rows', lambda: X41_train.shape, (120, 4))
check('4.1 test rows', lambda: X41_test.shape, (30, 4))
check('4.1 the test flowers are the ones random_state=20 picks',
      lambda: sorted(X41_test.index), TEST_ROWS_41)
check('4.1 every flower is in exactly one set',
      lambda: sorted(list(X41_train.index) + list(X41_test.index)) == sorted(X_AT_START.index), True)
check('4.1 every label still belongs to its flower',
      lambda: bool(y41_train.equals(Y_AT_START.loc[X41_train.index])
                   and y41_test.equals(Y_AT_START.loc[X41_test.index])), True)
check('4.1 the features are untouched',
      lambda: bool(X41_train.equals(X_AT_START.loc[X41_train.index])
                   and X41_test.equals(X_AT_START.loc[X41_test.index])), True)

**Task 4.2.** Fit a `KNeighborsClassifier` with K = 5 on the training set. Store the
fitted model in `m42`, and its accuracy on the **test** set, rounded to 4 decimals, in
`t42_acc`.

In [ ]:
m42 = KNeighborsClassifier(n_neighbors=5).fit(X41_train, y41_train)
t42_acc = round(m42.score(X41_test, y41_test), 4)

In [ ]:
check('4.2 K is 5', lambda: m42.n_neighbors, 5)
check('4.2 fitted on the 120 training flowers', lambda: m42.n_samples_fit_, 120)
check('4.2 test accuracy', lambda: round(float(t42_acc), 4), 0.9667)

**Task 4.3.** Find the mistake. Predict the test set with `m42`, then use a boolean mask
(predictions != true labels) to find:

- `t43_n_wrong` – how many test flowers were misclassified
- `t43_true` – the true species **name** of the misclassified flower
- `t43_pred` – the species **name** the model predicted for it

In [ ]:
preds_43 = m42.predict(X41_test)
wrong_mask = (preds_43 != y41_test)
t43_n_wrong = int(wrong_mask.sum())
t43_true = str(iris.target_names[y41_test[wrong_mask].iloc[0]])
t43_pred = str(iris.target_names[preds_43[wrong_mask][0]])

In [ ]:
check('4.3 how many wrong', lambda: whole(t43_n_wrong), 1)
check('4.3 its true species', lambda: str(t43_true), 'versicolor')
check('4.3 what the model called it', lambda: str(t43_pred), 'virginica')

**Task 4.4.** A new flower measures sepal length 6.7, sepal width 3.0, petal length 5.2,
petal width 2.3 (all in cm). Build it as a **one-row DataFrame with the same columns as
`X`**, predict its species with `m42`, and store the species **name** in `t44_name`.

In [ ]:
new_flower_df = pd.DataFrame([[6.7, 3.0, 5.2, 2.3]], columns=X.columns)
t44_name = str(iris.target_names[m42.predict(new_flower_df)[0]])

In [ ]:
check('4.4 species of the new flower', lambda: str(t44_name), 'virginica')

**Task 4.5.** The shape trap. Use the `raises` helper to find the **name of the exception**
you get when you pass one flower as a flat list:

```python
t45_error = raises(lambda: m42.predict([6.7, 3.0, 5.2, 2.3]))
```

The first line of the answer cell prints the whole message with `error_message`. Read it.

In [ ]:
# explain: prints the whole error message, without stopping the notebook
print(error_message(lambda: m42.predict([6.7, 3.0, 5.2, 2.3])))

t45_error = raises(lambda: m42.predict([6.7, 3.0, 5.2, 2.3]))

In [ ]:
check('4.5 exception name', t45_error, 'ValueError')

---
## Part 5 – Choosing K  ·  §7.5

Use the split from Task 4.1 throughout this part.

**Task 5.1 – predict first.** A K = 1 model is scored twice: on its own training flowers,
and on the test flowers. Before you run anything, write down which you expect to be higher.

Then fit `KNeighborsClassifier(n_neighbors=1)` on the training set and store:

- `t51_train` – its training accuracy
- `t51_test` – its test accuracy
- `t51_higher` – `'train'` or `'test'`, whichever came out higher

In [ ]:
# Prediction before running: training accuracy will be higher (1.0 on training vs lower on test).
m51 = KNeighborsClassifier(n_neighbors=1).fit(X41_train, y41_train)
t51_train = m51.score(X41_train, y41_train)
t51_test = m51.score(X41_test, y41_test)
t51_higher = 'train' if t51_train > t51_test else 'test'

In [ ]:
check('5.1 K = 1 training accuracy', lambda: round(float(t51_train), 4), 1.0)
check('5.1 K = 1 test accuracy', lambda: round(float(t51_test), 4), 0.9)
check('5.1 which is higher', t51_higher, 'train')

**Task 5.2.** Compare K values on held-out data. For each K in `[1, 3, 5, 7, 9, 11, 13, 15]`,
fit on the training set and record the **test** accuracy, rounded to 4 decimals. Store
them as a list, in that K order, in `t52_test_acc`.

In [ ]:
t52_test_acc = [
    round(KNeighborsClassifier(n_neighbors=k).fit(X41_train, y41_train).score(X41_test, y41_test), 4)
    for k in [1, 3, 5, 7, 9, 11, 13, 15]
]

In [ ]:
check('5.2 test accuracy for K = 1, 3, ..., 15',
      lambda: [round(float(v), 4) for v in t52_test_acc],
      [0.9, 0.9333, 0.9667, 1.0, 1.0, 1.0, 1.0, 1.0])

**Task 5.3.** Choose K. Store in `t53_k` a K from the **middle of the plateau** of best
test scores in Task 5.2, and in `t53_by_training` the answer to: *should K be chosen by
training accuracy?* (`True` or `False`).

In [ ]:
t53_k = 11
t53_by_training = False

In [ ]:
check('5.3 K is on the plateau and not at its edge',
      lambda: isinstance(t53_k, (int, np.integer)) and not isinstance(t53_k, bool) and int(t53_k) in (9, 11, 13), True)
check('5.3 choose by training accuracy?', t53_by_training, False)

**Task 5.4.** Push the dial the other way. Fit K = 101 on the training set and store its
training and test accuracy, each rounded to 4 decimals, in `t54_train` and `t54_test`.
Then store the name of this failure, `'overfitting'` or `'underfitting'`, in `t54_name`.

In [ ]:
m54 = KNeighborsClassifier(n_neighbors=101).fit(X41_train, y41_train)
t54_train = round(m54.score(X41_train, y41_train), 4)
t54_test = round(m54.score(X41_test, y41_test), 4)
t54_name = 'underfitting'

In [ ]:
check('5.4 K = 101 training accuracy', lambda: round(float(t54_train), 4), 0.6583)
check('5.4 K = 101 test accuracy', lambda: round(float(t54_test), 4), 0.5667)
check('5.4 the name of the failure', t54_name, 'underfitting')

---
## Part 6 – Feature scaling  ·  §7.6

`customers` holds the four customers from §7.6, and `new_customer` the one who has not
decided. `members` holds 300 club members you have not seen before.

**Task 6.1.** Fit a K = 1 model on the **raw** `income_usd` and `age` columns of
`customers`, with `premium` as the label. Predict for `new_customer` and store the
answer (`'yes'` or `'no'`) in `t61_raw`.

In [ ]:
m61 = KNeighborsClassifier(n_neighbors=1).fit(customers[['income_usd', 'age']], customers['premium'])
t61_raw = m61.predict(new_customer)[0]

In [ ]:
check('6.1 raw 1-NN prediction', lambda: str(t61_raw), 'yes')

**Task 6.2.** Fit a `StandardScaler` on the four customers' two feature columns and store
it in `t62_scaler`. Transform both the customers and `new_customer` with it, fit a K = 1
model on the scaled customers, and store the new prediction in `t62_scaled`.

In [ ]:
t62_scaler = StandardScaler().fit(customers[['income_usd', 'age']])
m62 = KNeighborsClassifier(n_neighbors=1).fit(t62_scaler.transform(customers[['income_usd', 'age']]), customers['premium'])
t62_scaled = m62.predict(t62_scaler.transform(new_customer))[0]

In [ ]:
check('6.2 the scaler learned the four customers', lambda: [round(float(v), 2) for v in t62_scaler.mean_], [59500.0, 43.25])
check('6.2 scaled 1-NN prediction', lambda: str(t62_scaled), 'no')

**Task 6.3 – predict first.** In `members`, whether a person joined depends on their age.
Before you run anything, write down whether you expect scaling to change the test
accuracy of a K = 5 model, and by how much.

Split `members[['income_usd', 'age']]` and `members['joined']` with `test_size=0.25` and
`random_state=7`. Then store, each rounded to 4 decimals:

- `t63_raw` – the test accuracy of K = 5 on the raw features
- `t63_scaled` – the test accuracy of K = 5 after a `StandardScaler` fitted on the
  **training** rows only

Also store that scaler in `t63_scaler`.

In [ ]:
# Prediction before running: scaling will substantially increase test accuracy because age drives joining.
X63_train, X63_test, y63_train, y63_test = train_test_split(
    members[['income_usd', 'age']], members['joined'], test_size=0.25, random_state=7
)
t63_raw = round(KNeighborsClassifier(n_neighbors=5).fit(X63_train, y63_train).score(X63_test, y63_test), 4)
t63_scaler = StandardScaler().fit(X63_train)
t63_scaled = round(
    KNeighborsClassifier(n_neighbors=5).fit(t63_scaler.transform(X63_train), y63_train).score(
        t63_scaler.transform(X63_test), y63_test
    ), 4
)

In [ ]:
check('6.3 raw test accuracy', lambda: round(float(t63_raw), 4), 0.5867)
check('6.3 scaled test accuracy', lambda: round(float(t63_scaled), 4), 0.8933)
check('6.3 scaler fitted on the 225 training members only',
      lambda: bool(np.allclose(t63_scaler.mean_, train_test_split(
          MEMBERS_AT_START[['income_usd', 'age']], test_size=0.25, random_state=7)[0].mean().to_numpy())), True)

**Task 6.4.** The complete, correct sequence on Iris. Using the split from Task 4.1, fit a
`StandardScaler` on the **training** flowers only and store it in `t64_scaler`. Transform
both sets, fit K = 5 on the scaled training set, and store the test accuracy, rounded to 4
decimals, in `t64_acc`.

In [ ]:
t64_scaler = StandardScaler().fit(X41_train)
m64 = KNeighborsClassifier(n_neighbors=5).fit(t64_scaler.transform(X41_train), y41_train)
t64_acc = round(m64.score(t64_scaler.transform(X41_test), y41_test), 4)

In [ ]:
check('6.4 scaler fitted on the training flowers only',
      lambda: bool(np.allclose(t64_scaler.mean_, X41_train.mean().to_numpy())), True)
check('6.4 scaled test accuracy', lambda: round(float(t64_acc), 4), 0.9)

---
## Part 7 – Traps  ·  §7.8

**Task 7.1.** A test set has 200 customers; 8 cancelled (`1`) and 192 did not (`0`).
A lazy "model" predicts `0` for everyone. Build the true labels and the lazy predictions
as arrays, then store:

- `t71_acc` – the lazy model's accuracy
- `t71_caught` – how many of the 8 cancellations it caught

In [ ]:
y71_true = np.array([1] * 8 + [0] * 192)
y71_pred = np.zeros(200, dtype=int)
t71_acc = float((y71_pred == y71_true).mean())
t71_caught = int(((y71_pred == 1) & (y71_true == 1)).sum())

In [ ]:
check('7.1 lazy accuracy', lambda: round(float(t71_acc), 4), 0.96)
check('7.1 cancellations caught', lambda: whole(t71_caught), 0)

**Task 7.2.** Each scenario ends with a **claim**. Answer `'trustworthy'` if the evidence
supports the claim, or `'not trustworthy'` if it does not:

- `t72_a` – K = 1 had the best **training** accuracy. Claim: K = 1 will do best on new data.
- `t72_b` – a scaler fitted on the training rows, K = 5, scored once on held-back rows.
  Claim: this score estimates accuracy on new data.
- `t72_c` – a scaler fitted on **all** the rows before splitting, then scored on the test rows.
  Claim: this score is an honest estimate.
- `t72_d` – accuracy 0.96 on a test set where 96% of the rows belong to one class.
  Claim: the model is good at finding the rare class.

In [ ]:
t72_a = 'not trustworthy'
t72_b = 'trustworthy'
t72_c = 'not trustworthy'
t72_d = 'not trustworthy'

In [ ]:
check('7.2 a – K chosen by training accuracy', t72_a, 'not trustworthy')
check('7.2 b – the correct sequence', t72_b, 'trustworthy')
check('7.2 c – scaler fitted before the split', t72_c, 'not trustworthy')
check('7.2 d – 0.96 on 96% one class', t72_d, 'not trustworthy')

**Task 7.3 – GUARD.** Nothing to write.

The check below reads `X_six`, `y_six`, `X`, `y`, `customers` and `members` directly and
confirms they still hold what the setup cell put there. It passes on an untouched notebook,
so it is not evidence of work – it exists to fail loudly if something above overwrote the
shared data.

In [ ]:
# Nothing to write for this task.

In [ ]:
check('7.3 GUARD X_six untouched', lambda: bool(np.array_equal(X_six, X_SIX_AT_START)), True)
check('7.3 GUARD y_six untouched', lambda: bool(np.array_equal(y_six, Y_SIX_AT_START)), True)
check('7.3 GUARD X untouched', lambda: X.equals(X_AT_START), True)
check('7.3 GUARD y untouched', lambda: y.equals(Y_AT_START), True)
check('7.3 GUARD customers untouched', lambda: customers.equals(CUSTOMERS_AT_START), True)
check('7.3 GUARD members untouched', lambda: members.equals(MEMBERS_AT_START), True)

---
## Finish

Choose **Kernel → Restart Kernel and Run All Cells** one last time. If every check prints `PASS` from a clean start, your work is reproducible and you are done.

**Two answers to carry into the quiz:**

- A perfect training score from K = 1 comes from the arithmetic, not from a good model (Task 5.1). Choose K on held-out data.
- When features are on very different scales, the large one decides the distance. Scale first, with a scaler fitted on the training rows only (Tasks 6.3 and 6.4).

---
## Your submission line

Run the cell below and paste **the line it prints** into *Lab 7 – submit* on D2L. Nothing else – not the address bar of this notebook, which carries a port number that stops working when your server restarts. The line is built from your account name, so it always works.

In [1]:
# Run this cell. Copy the line it prints into 'Lab 7 -- submit' on D2L. Nothing else.
from pathlib import Path
account = Path.home().name                   # your gblearn account name
print(f'http://{account}.gblearn.com/python/?open=comp3122/labs/lab07/COMP3122-Lab07-Student.ipynb')

http://f6172332.gblearn.com/python/?open=comp3122/labs/lab07/COMP3122-Lab07-Student.ipynb
